<a href="https://colab.research.google.com/github/hollyemblem/out-of-domain-mmd/blob/mmd-experiments/domain_shift_experiments_jigsaw_tweet_evalmmd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MMD Experiments

### Method:

"Intuitively, the MMD test evaluates whether there is a significant difference between two distributions: a higher MMD value suggests a greater disparity between the distributions. The test essentially disproves the null hypothesis that the distributions are identical—when the MMD statistic is significantly high. In our study, the MMD values can appear negative due to estimation errors in smaller samples or due to the kernel choice affecting the calculation. However, the absolute value of MMD should be considered. Typically, a threshold for significance is set, above which the null hypothesis can be rejected. We use 0.05 as our threshold."

Source: https://link.springer.com/article/10.1007/s10579-024-09754-8#Sec3


### Dataset trials

- Jigsaw 50/50 split (should not be OOD)
- Perspective’s OOD example example ( TweetEval) <- This is what we're trialling
- Jigsaw data + I2P benchmark
- Jigsaw data + DiffusionDB

Source: https://arxiv.org/pdf/2202.11176

### GPU Setup

In [1]:
import torch

torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
print(torch.cuda.is_available())

True


#### Installing Required Libraries


In [8]:
# @title
#!pip install -U sentence-transformers
#https://huggingface.co/efederici/sentence-bert-base



In [9]:
!pip install pytorch-ignite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 371.8/371.8 kB 15.6 MB/s eta 0:00:00


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:

%cd /content/drive/My Drive/Colab Notebooks/domain_shift_experiments_05_2026

/content/drive/My Drive/Colab Notebooks/domain_shift_experiments_05_2026


In [12]:
%pip install -q --upgrade kagglehub

## Import Libraries

In [14]:
from google.colab import userdata
import pandas as pd
import kagglehub
import os
import requests
import numpy as np
from sentence_transformers import SentenceTransformer
from ignite.metrics import MaximumMeanDiscrepancy

In [15]:
KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_KEY = userdata.get('KAGGLE_KEY')

In [16]:
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [17]:

path = kagglehub.competition_download(
    "jigsaw-unintended-bias-in-toxicity-classification"
)

print("Path to competition files:", path)

100%|██████████| 723M/723M [00:20<00:00, 37.0MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/jigsaw-unintended-bias-in-toxicity-classification


In [18]:
print(os.listdir(path))

['train.csv', 'identity_individual_annotations.csv', 'all_data.csv', 'sample_submission.csv', 'test_private_expanded.csv', 'toxicity_individual_annotations.csv', 'test_public_expanded.csv', 'test.csv']


In [19]:
train_path = os.path.join(path, "train.csv")
training = pd.read_csv(train_path)
print(training.shape)
training.head()

(1804874, 45)


,id,target,comment_text,severe_toxicity,obscene,identity_attack,insult,threat,asian,atheist,...,article_id,rating,funny,wow,sad,likes,disagree,sexual_explicit,identity_annotator_count,toxicity_annotator_count
0,59848,0.000000,"This is so cool. It's like, 'would you want yo...",0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
1,59849,0.000000,Thank you!! This would make my life a lot less...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
2,59852,0.000000,This is such an urgent design problem; kudos t...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
3,59855,0.000000,Is this something I'll be able to install on m...,0.000000,0.0,0.000000,0.00000,0.0,NaN,NaN,...,2006,rejected,0,0,0,0,0,0.0,0,4
4,59856,0.893617,haha you guys are a bunch of losers.,0.021277,0.0,0.021277,0.87234,0.0,0.0,0.0,...,2006,rejected,0,0,0,1,0,0.0,4,47


### TweetEval Dataset
"We evaluate our models on the TweetEval hate content
classification test split [3], which is taken from the SemEval2019
Hateval challenge [4]. The task is to predict whether a given tweet
contains hateful language targeted against any of two communities: women and immigrants. This task differs from our UTC
pre-training and fine-tuning as it is purely Tweet focused and has
been labeled to a different standard (i.e. hateful language.) As this
is zero-shot evaluation, we do not do any additional fine-tuning for
this experiment."

### Loading TweetEval Hate Test Data

In [20]:
#https://github.com/cardiffnlp/tweeteval
# URL of the raw test.txt file on GitHub
tweeteval_hate_test_url = "https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/hate/test_text.txt"

# Download the file
response = requests.get(tweeteval_hate_test_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

# Save the content to a local file
with open("tweeteval_hate_test.txt", "w", encoding="utf-8") as f:
    f.write(response.text)

print("Downloaded tweeteval_hate_test.txt")

Downloaded tweeteval_hate_test.txt


In [21]:
# Load the test data into a list
tweet_eval_test_texts = []
with open("tweeteval_hate_test.txt", "r", encoding="utf-8") as f:
    for line in f:
        tweet_eval_test_texts.append(line.strip())

print(f"Loaded {len(tweet_eval_test_texts)} lines from tweeteval_hate_test.txt")

Loaded 2970 lines from tweeteval_hate_test.txt


In [22]:
# Convert the list of texts to a pandas Series
tweet_series = pd.Series(tweet_eval_test_texts)
tweet_series.head()

,0
0,"@user , you are correct that Reid certainly is..."
1,Whoever just unfollowed me you a bitch
2,@user @user Those People Invaded Us!!! They DO...
3,"stop JUDGING bitches by there cover, jus cuz s..."
4,how about i knock heads off and send them gift...


In [23]:
split_by_comma_space = tweet_series.str.split("', ").str[0]
split_by_comma_space.head()

,0
0,"@user , you are correct that Reid certainly is..."
1,Whoever just unfollowed me you a bitch
2,@user @user Those People Invaded Us!!! They DO...
3,"stop JUDGING bitches by there cover, jus cuz s..."
4,how about i knock heads off and send them gift...


In [24]:
# Calculate the sum of lengths for the original tweet_series
original_series_length_sum = tweet_series.str.len().sum()
print(f"Sum of character lengths for original tweet_series: {original_series_length_sum}")

Sum of character lengths for original tweet_series: 390761


In [25]:
tweeteval = tweet_series.to_list()

In [26]:
jigsaw_data = training.sample(n=len(tweeteval))

In [27]:
jigsaw_list = jigsaw_data['comment_text'].to_list()

### Sentence Embeddings Setup

In [28]:
print(torch.cuda.get_device_name(0))

model = SentenceTransformer(
    'efederici/sentence-bert-base',
    device='cuda'
)

print(model.device)

NVIDIA A100-SXM4-80GB


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.43k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/725k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

cuda:0


In [29]:
##train embeddings
model = model.to(device)

train_embeddings = model.encode(
    jigsaw_list,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(train_embeddings.shape)

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

(2970, 768)


In [30]:
##val embeddings

val_embeddings = model.encode(
    tweeteval,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(val_embeddings.shape)

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

(2970, 768)


In [31]:
##Maximum Mean Discrepancy https://docs.pytorch.org/ignite/generated/ignite.metrics.MaximumMeanDiscrepancy.html

In [38]:
#Example code from https://github.com/emanuele/kernel_two_sample_test/blob/master/kernel_two_sample_test.py
from __future__ import division
import numpy as np
from sys import stdout
from sklearn.metrics import pairwise_kernels


In [42]:
#From: https://github.com/emanuele/kernel_two_sample_test/blob/master/kernel_two_sample_test.py
def MMD2u(K, m, n):
    """The MMD^2_u unbiased statistic.
    """
    Kx = K[:m, :m]
    Ky = K[m:, m:]
    Kxy = K[:m, m:]
    return 1.0 / (m * (m - 1.0)) * (Kx.sum() - Kx.diagonal().sum()) + \
        1.0 / (n * (n - 1.0)) * (Ky.sum() - Ky.diagonal().sum()) - \
        2.0 / (m * n) * Kxy.sum()


def compute_null_distribution(K, m, n, iterations=10000, verbose=False,
                              random_state=None, marker_interval=1000):
    """Compute the bootstrap null-distribution of MMD2u.
    """
    if type(random_state) == type(np.random.RandomState()):
        rng = random_state
    else:
        rng = np.random.RandomState(random_state)

    mmd2u_null = np.zeros(iterations)
    for i in range(iterations):
        if verbose and (i % marker_interval) == 0:
            print(i),
            stdout.flush()
        idx = rng.permutation(m+n)
        K_i = K[idx, idx[:, None]]
        mmd2u_null[i] = MMD2u(K_i, m, n)

    if verbose:
        print("")

    return mmd2u_null


def compute_null_distribution_given_permutations(K, m, n, permutation,
                                                 iterations=None):
    """Compute the bootstrap null-distribution of MMD2u given
    predefined permutations.

    Note:: verbosity is removed to improve speed.
    """
    if iterations is None:
        iterations = len(permutation)

    mmd2u_null = np.zeros(iterations)
    for i in range(iterations):
        idx = permutation[i]
        K_i = K[idx, idx[:, None]]
        mmd2u_null[i] = MMD2u(K_i, m, n)

    return mmd2u_null


def kernel_two_sample_test(X, Y, kernel_function='rbf', iterations=10000,
                           verbose=False, random_state=None, **kwargs):
    """Compute MMD^2_u, its null distribution and the p-value of the
    kernel two-sample test.

    Note that extra parameters captured by **kwargs will be passed to
    pairwise_kernels() as kernel parameters. E.g. if
    kernel_two_sample_test(..., kernel_function='rbf', gamma=0.1),
    then this will result in getting the kernel through
    kernel_function(metric='rbf', gamma=0.1).
    """
    m = len(X)
    n = len(Y)
    XY = np.vstack([X, Y])
    K = pairwise_kernels(XY, metric=kernel_function, **kwargs)
    mmd2u = MMD2u(K, m, n)
    if verbose:
        print("MMD^2_u = %s" % mmd2u)
        print("Computing the null distribution.")

    mmd2u_null = compute_null_distribution(K, m, n, iterations,
                                           verbose=verbose,
                                           random_state=random_state)
    p_value = max(1.0/iterations, (mmd2u_null > mmd2u).sum() /
                  float(iterations))
    if verbose:
        print("p-value ~= %s \t (resolution : %s)" % (p_value, 1.0/iterations))

    return mmd2u, mmd2u_null, p_value

In [43]:
# Using https://github.com/emanuele/kernel_two_sample_test/blob/master/kernel_two_sample_test.py to calculate mmu^2 unbiased as per paper
#MMD usage paper = https://link.springer.com/article/10.1007/s10579-024-09754-8#Sec3
from scipy.spatial.distance import pdist
from sklearn.metrics.pairwise import pairwise_kernels
x = torch.tensor(train_embeddings, dtype=torch.float32, device=device).detach().cpu().numpy()
y = torch.tensor(val_embeddings, dtype=torch.float32, device=device).detach().cpu().numpy()

m = len(x)
n = len(y)

# Pool the distributions
Z = np.vstack([x, y])

# sigma = Calculate the median Euclidean distance between points
sigma = np.median(
    pdist(Z, metric="euclidean") #Review paper here: https://www.jmlr.org/papers/volume13/gretton12a/gretton12a.pdf
)

print("sigma:", sigma)

# sklearn RBF:
# exp(-gamma * ||x-y||²)
#
# Paper:
# exp(-(1/sigma) * ||x-y||²)
#
# Therefore gamma = 1/sigma

K = pairwise_kernels(
    Z,
    metric="rbf",
    gamma=1.0 / sigma
)

mmd2_u = MMD2u(K, m, n)

print("Unbiased MMD²:", mmd2_u)

sigma: 0.9633299688199457
Unbiased MMD²: 0.10530299


In [44]:
kernel_two_sample_test(x,y,kernel_function="rbf",
    gamma=1.0 / sigma,
    iterations=10000,
    random_state=1,
    verbose=True)

(np.float32(0.00031244755),
 array([ 2.14576721e-06, -3.57627869e-07,  2.38418579e-07, ...,
         2.38418579e-07, -3.57627869e-07, -1.31130219e-06]),
 0.0001)